In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin, Affine
from pathlib import Path
import xarray as xr
import pdb
from pyproj import Transformer

def get_lon_lat(ds):
    lon_names = ["lon", "longitude", "x"]
    lat_names = ["lat", "latitude", "y"]

    lon = next((ds[v] for v in lon_names if v in ds), None)
    lat = next((ds[v] for v in lat_names if v in ds), None)

    if lon is None or lat is None:
        raise ValueError("Longitude/Latitude not found in dataset")

    return lon, lat

In [ ]:
region = "Atka_Bay"

In [ ]:
nc_dir = Path(f"/g/data/gv90/ae8456/SWOT_SEAICE/DATA/{region}/SWOT_FREEBOARD_mosaic_cycle_new{region}")
out_dir = Path(
    f"/g/data/gv90/ae8456/SWOT_SEAICE/DATA/{region}/SWOT_FREEBOARD_mosaic_cycle_new{region}/output_geotiffs"
)
out_dir.mkdir(parents=True, exist_ok=True)

nc_files = sorted(nc_dir.rglob("*.nc"))

In [ ]:
nc_files

## Single orbit Work in Progress

## Mosaics 

In [ ]:
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from pyproj import Transformer

# Known grid resolution (meters)
PIXEL_SIZE = 250.0

# Forward projection: lat/lon → EPSG:3031
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3031",
    always_xy=True
)

# now process nc_file
#process_nc(nc_file)
for nc_file in nc_files:

    print(f"Processing {nc_file.name}")
    ds = xr.open_dataset(nc_file)



    nc_out_dir = out_dir / nc_file.stem
    nc_out_dir.mkdir(parents=True, exist_ok=True)

    lon, lat = get_lon_lat(ds)

    # Handle 1D or 2D lon/lat
    if lon.ndim == 1 and lat.ndim == 1:
        lon2d, lat2d = np.meshgrid(lon.values, lat.values)
    else:
        lon2d, lat2d = lon.values, lat.values

    # Project to EPSG:3031
  
    X, Y = transformer.transform(lon2d, lat2d)

    # Build transform using known pixel size
    transform = from_origin(
        X.min(),        # west
        Y.max(),        # north
        PIXEL_SIZE,     # dx
        PIXEL_SIZE      # dy
    )

    for var_name, da in ds.data_vars.items():

        if var_name.lower() in ["lon", "longitude", "lat", "latitude"]:
            continue

        if not set(da.dims).issuperset(set(lat.dims + lon.dims)):
            continue

        if "time" in da.dims:
            for t in range(da.sizes["time"]):
                data = da.isel(time=t).values.T
                data = np.flipud(data)
                timestamp = str(da.time.values[t]).replace(":", "").replace("-", "")
                out_path = nc_out_dir / f"{var_name}_{timestamp}.tif"

                with rasterio.open(
                    out_path,
                    "w",
                    driver="GTiff",
                    height=data.shape[0],
                    width=data.shape[1],
                    count=1,
                    dtype=data.dtype,
                    crs="EPSG:3031",
                    transform=transform,
                ) as dst:
                    dst.write(data, 1)

        else:
            data = da.values.T
            data = np.flipud(data)
            out_path = nc_out_dir / f"{var_name}.tif"

            with rasterio.open(
                out_path,
                "w",
                driver="GTiff",
                height=data.shape[0],
                width=data.shape[1],
                count=1,
                dtype=data.dtype,
                crs="EPSG:3031",
                transform=transform,
            ) as dst:
                dst.write(data, 1)

    ds.close()